# Cross-validation and business-cost analysis

This notebook summarizes five-fold stratified cross-validation for every configured model. We compare mean performance, fold-to-fold variability, aggregated confusion matrices, and a maintenance-oriented error cost.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'reports').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

results = pd.read_csv(PROJECT_ROOT / 'reports' / 'cross_validation_results.csv')
confusion_payload = json.loads((PROJECT_ROOT / 'reports' / 'cross_validation_confusion_matrices.json').read_text())
labels = confusion_payload['labels']
confusion_matrices = confusion_payload['matrices']
display(results.round(4))

In [ ]:
metric_columns = [
    'model', 'accuracy_mean', 'accuracy_std', 'macro_f1_mean', 'macro_f1_std',
    'repair_recall_mean', 'repair_recall_std', 'business_cost_mean', 'business_cost_std'
]
display(results[metric_columns].sort_values('business_cost_mean').round(4))

In [ ]:
top_models = results.nsmallest(4, 'business_cost_mean')['model'].tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, model_name in zip(axes.flat, top_models):
    matrix = pd.DataFrame(confusion_matrices[model_name], index=labels, columns=labels)
    normalized = matrix.div(matrix.sum(axis=1), axis=0)
    sns.heatmap(normalized, annot=True, fmt='.2f', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(model_name)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
plt.tight_layout()
plt.show()

## Decision framework

The business-cost column assigns five times more penalty to missing a pump that needs repair than to another classification error. This is a transparent starting assumption, not a measured operational contract. The portfolio demo exposes several trained models so the trade-off between overall performance and maintenance recall remains visible.